In [17]:
# === DATA CLEANING: KAGGLE DATASET ===
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [18]:
print("=== 1. LOAD DATA ===")
df = pd.read_csv("predictive_maintenance_dataset.csv")
print("Original Shape:", df.shape)

=== 1. LOAD DATA ===
Original Shape: (124494, 12)


In [19]:
print("\n=== 2. CLEAN COLUMN NAMES ===")
df.columns = df.columns.str.strip().str.lower()
df.columns = [re.sub(r'[^a-z0-9_]', '_', col) for col in df.columns]
print("Columns:", df.columns.tolist())


=== 2. CLEAN COLUMN NAMES ===
Columns: ['date', 'device', 'failure', 'metric1', 'metric2', 'metric3', 'metric4', 'metric5', 'metric6', 'metric7', 'metric8', 'metric9']


In [20]:
print("\n=== 3. CLEAN DATES ===")
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df = df.sort_values('date').reset_index(drop=True)


=== 3. CLEAN DATES ===


In [21]:
print("\n=== 4. MISSING VALUES ===")
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


=== 4. MISSING VALUES ===


In [22]:
print("\n=== 5. REMOVE DUPLICATES ===")
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicates")


=== 5. REMOVE DUPLICATES ===
Removed 1 duplicates


In [23]:
print("\n=== 6. RENAME METRICS + CONVERT TO KELVIN ===")

# RENAME METRICS TO REAL NAMES
df = df.rename(columns={
    'metric1': 'Air_Temp_C',      
    'metric2': 'Process_Temp_C',  
    'metric3': 'Rotational_Speed', 
    'metric4': 'Torque',
    'metric5': 'Tool_Wear',
    'metric6': 'Pressure',
    'metric7': 'Vibration',
    'metric8': 'Voltage',
    'metric9': 'Current'
      })
print("New Columns:", df.columns.tolist())

    # CONVERT C TO KELVIN: K = C + 273.15
df['Air_Temp_K'] = df['Air_Temp_C'] + 273.15
df['Process_Temp_K'] = df['Process_Temp_C'] + 273.15

# ADD ENGINEERING LIMIT FLAGS
df['Air_Temp_Alert'] = np.where(df['Air_Temp_K'] > 320, 1, 0)      # > 47C
df['Process_Temp_Alert'] = np.where(df['Process_Temp_K'] > 350, 1, 0) # > 77C
df['Critical_Temp_Flag'] = np.where(df['Process_Temp_K'] > 373, 1, 0) # > 100C = 373K
df['RPM_Alert'] = np.where(df['Rotational_Speed'] > 4000, 1, 0)
df['Tool_Wear_Alert'] = np.where(df['Tool_Wear'] > 200, 1, 0)

print("Added Kelvin columns and Alert flags")



=== 6. RENAME METRICS + CONVERT TO KELVIN ===
New Columns: ['date', 'device', 'failure', 'Air_Temp_C', 'Process_Temp_C', 'Rotational_Speed', 'Torque', 'Tool_Wear', 'Pressure', 'Vibration', 'Voltage', 'Current']
Added Kelvin columns and Alert flags


In [24]:
print("\n=== 7. FINAL ===")
print("Cleaned Shape:", df.shape)
print("Missing:", df.isnull().sum().sum())
print(df.head())
df.to_csv("cleaned_predictive_maintenance_dataset.csv", index=False)
print("\n✅ Saved as: cleaned_predictive_maintenance_dataset.csv")


=== 7. FINAL ===
Cleaned Shape: (124493, 19)
Missing: 0
        date    device  failure  Air_Temp_C  Process_Temp_C  Rotational_Speed  \
0 2015-01-01  S1F01085        0   215630672              55                 0   
1 2015-01-01  W1F0Y13C        0   234318640               0                 0   
2 2015-01-01  W1F0XKWR        0    89660704               0                 0   
3 2015-01-01  W1F0X7QX        0   162013456               0                 0   
4 2015-01-01  W1F0X7PR        0    13138392               0                 0   

   Torque  Tool_Wear  Pressure  Vibration  Voltage  Current    Air_Temp_K  \
0      52          6    407438          0        0        7  2.156309e+08   
1       0          4    185772          0        0        3  2.343189e+08   
2       0          7        30          0        0        0  8.966098e+07   
3       0         12    217686          0        0        0  1.620137e+08   
4       0          9    191343          0        0        0  1.313867e+